# Architecture registration

> One object tells MAX: *this is SmiTedModel — here's the tokenizer, batcher, graph, and weights.*


In [ ]:
#| default_exp arch


In [ ]:
#| hide
from nbdev.showdoc import *


## The wiring diagram

When you run:

```bash
max serve \
  --model-path ./model_assets/ibm-research_materials.smi-ted \
  --custom-architectures ./mat_gram01 \
  --quantization-encoding float32
```

MAX imports the package, reads `ARCHITECTURES`, and matches `config.json`'s `"architectures": ["SmiTedModel"]` to `smi_ted_arch.name`.

Everything we've built plugs into one `SupportedArchitecture`:


In [ ]:
#| export
from max.graph.weights import WeightsFormat
from max.pipelines.context import TextContext
from max.pipelines.lib import SupportedArchitecture
from max.pipelines.modeling.types import PipelineTask

from mat_gram01 import weight_adapters
from mat_gram01.batch_processor import SmiTedBatchProcessor
from mat_gram01.model import SmiTedInputs, SmiTedPipelineModel
from mat_gram01.model_config import SmiTedModelConfig
from mat_gram01.tokenizer import SmiTedTokenizer


In [ ]:
#| export
smi_ted_arch = SupportedArchitecture(
    name="SmiTedModel",
    task=PipelineTask.EMBEDDINGS_GENERATION,
    example_repo_ids=[
        "ibm-research/materials.smi-ted",
    ],
    default_encoding="float32",
    supported_encodings={
        "float32",
        "bfloat16",
    },
    pipeline_model=SmiTedPipelineModel,
    tokenizer=SmiTedTokenizer,
    context_type=TextContext,
    default_weights_format=WeightsFormat.safetensors,
    weight_adapters={
        WeightsFormat.safetensors: weight_adapters.convert_safetensor_state_dict,
    },
    required_arguments={"enable_prefix_caching": False},
    config=SmiTedModelConfig,
    batching=SmiTedBatchProcessor,
)


MAX discovers custom architectures via a package-level ``ARCHITECTURES`` list. We define it next to the arch object so both stay in sync.


In [ ]:
#| export
ARCHITECTURES = [smi_ted_arch]


| Field | Points to |
|-------|-----------|
| `name` | `"SmiTedModel"` (must match config) |
| `task` | `EMBEDDINGS_GENERATION` |
| `pipeline_model` | `SmiTedPipelineModel` |
| `tokenizer` | `SmiTedTokenizer` |
| `batching` | `SmiTedBatchProcessor` |
| `config` | `SmiTedModelConfig` |
| `weight_adapters` | safetensors → filtered state dict |


### Checkpoint

Explain how `--custom-architectures` + `config.json` `"architectures"` select this package.


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()
